In [33]:
import os
import re
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

In [34]:
# Config 
DATA_PATH = "raw_data/Sarcasm_Headlines_Dataset_v2.json"   
OUT_DIR = "processed_data/sarcasm_news"                               
RANDOM_SEED = 42                                          
TEST_SIZE = 0.20                                           
VAL_SIZE = 0.10     

In [35]:
#Load Data
data = pd.read_json(DATA_PATH, lines=True)
data = data[['headline', 'is_sarcastic']] #article_link not needed
data = data.rename(columns={'headline': 'text', 'is_sarcastic': 'label'})

print(f"Total samples: {len(data)}\n")
print(data['label'].value_counts())

print(data.head())

Total samples: 28619

label
0    14985
1    13634
Name: count, dtype: int64
                                                text  label
0  thirtysomething scientists unveil doomsday clo...      1
1  dem rep. totally nails why congress is falling...      0
2  eat your veggies: 9 deliciously different recipes      0
3  inclement weather prevents liar from getting t...      1
4  mother comes pretty close to using word 'strea...      1


In [36]:
#clean
def remove_html(text):
    """Strip HTML tags using BeautifulSoup."""
    return BeautifulSoup(text, "html.parser").get_text()

def normalize_whitespace(text):
    """Normalize multiple spaces/newlines to single space."""
    return re.sub(r"\s+", " ", text).strip()

def clean_text(text):
    """Minimal cleaning: remove HTML + normalize whitespace only."""
    if not isinstance(text, str):
        return ""
    text = remove_html(text)
    text = normalize_whitespace(text)
    return text  # keep punctuation, casing, contractions (important for sarcasm)

data['text'] = data['text'].apply(clean_text)
data = data[data['text'].str.strip() != ""].reset_index(drop=True) # remove empty entries
data = data.drop_duplicates(subset="text").reset_index(drop=True) # remove duplicates

print(f"Samples after cleaning: {len(data)}\n")
print(data['label'].value_counts())

Samples after cleaning: 28503

label
0    14951
1    13552
Name: count, dtype: int64


In [37]:
#test-train-val split

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=data['label']
)

train_data, val_data = train_test_split(
    train_data,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_data['label']
)

print("\nFinal splits:")
print("Train:", len(train_data))
print("Val  :", len(val_data))
print("Test :", len(test_data))


Final splits:
Train: 20521
Val  : 2281
Test : 5701


In [39]:
#save processed data
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

train_data.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
val_data.to_csv(os.path.join(OUT_DIR, "val.csv"), index=False)
test_data.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)

data.to_csv(os.path.join(OUT_DIR, "sarcasm_Headlines_Dataset_cleaned.csv"), index=False)